<a href="https://colab.research.google.com/github/ishitamunshi/world_model_stacking/blob/main/notebooks/action_smoothness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 6280 - Experiment 3: Action Smoothness Cost

**Goal:** Test whether adding a smoothness penalty to world-model candidate scoring improves dexterous box stacking. The policy, world model, simulator, subgoals, and candidate count are unchanged.

**How to use:** In Colab select **Runtime → Change runtime type → T4 GPU**. Execute each cell from top to bottom. The evaluation cell executes 20 episodes and saves each completed run to Google Drive, so you can resume safely. Do not use Run All accidentally until you are ready to begin all evaluations.

**Development scenes:** Read the same five scene IDs as Experiments 1 and 2 from Google Drive; exclude held-out test scenes.

**Why rerun beta=0:** Previous baseline CSVs did not record actual executed-action smoothness. The new instrumented baseline makes comparisons fair; its peak/final results are checked against the saved baseline.

**Scoring:** `terminal_cost + beta * scale * smoothness`, with `scale = mean(terminal_cost)/max(mean(smoothness),1e-6)` in each candidate set. Smoothness uses the physical 25-dimensional raw commands after checkpoint unnormalization and clipping, including the boundary from the last executed command where available. It is not a measure of robot physical jerk.


## 1. Install requirements and check T4

In [1]:
import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path
import torch

try:
    import google.colab
except ImportError as e:
    raise RuntimeError("This notebook is designed to run on Google Colab.") from e

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU, then run this cell.")

# Matching the official planning notebook's system setup.
subprocess.check_call(["apt-get", "update", "-qq"],
                      env=dict(os.environ, DEBIAN_FRONTEND="noninteractive"))
subprocess.check_call(["apt-get", "install", "-y", "-qq",
                       "libegl1", "libgl1", "libgles2", "libglib2.0-0", "ffmpeg"],
                      env=dict(os.environ, DEBIAN_FRONTEND="noninteractive"))

packages = []
for module, requirement in [
    ("mujoco", "mujoco>=3.3,<4"),
    ("imageio_ffmpeg", "imageio[ffmpeg]>=2.30"),
    ("huggingface_hub", "huggingface_hub>=0.26"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
]:
    try:
        __import__(module)
    except ImportError:
        packages.append(requirement)
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


## 2. Load simulator and verify rendering

In [ ]:
# Official starter code and MuJoCo assets.
import os, sys, tarfile
from pathlib import Path
from huggingface_hub import hf_hub_download

ROOT = Path('/content/world-model-gym')
marker = ROOT/'assignment'/'training'/'planning'/'agent.py'
if not marker.exists():
    archive = hf_hub_download('YongYong/cis6280-stack-world-models-v2',
                              'code/world-model-gym.tar', repo_type='model')
    with tarfile.open(archive) as tar:
        tar.extractall('/content', filter='data')
assert marker.exists(), f"Missing code: {marker}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
hand = ROOT/'assets'/'sharpa'/'wave_01'/'right_sharpa_wave'/'right_sharpa_wave.xml'
if not hand.exists():
    subprocess.check_call([sys.executable, 'scripts/fetch_assets.py'])

from assignment.training.data.task import StackTaskEnv, TaskConfig
_env = StackTaskEnv(TaskConfig(target_count=4, horizon=10, terminate_on_ground=False), 100038)
try:
    _env.reset(('init',100038))
    _env.env.resolution=128
    print('Rendered camera shape:', list(_env.env.render())[0].shape)
finally:
    _env.close()


## 3. Download fixed world model, policy, and subgoals

In [ ]:
# Same pretrained checkpoints and goal dataset as prior experiments.
import shutil
import torch
from huggingface_hub import hf_hub_download

CKPT = Path('/content/stack-baseline')
CKPT.mkdir(parents=True, exist_ok=True)
WM = CKPT/'smwm-c3.pt'
POLICY = CKPT/'df-smwm-c1-sigma04.pt'
GOALS = CKPT/'goals-every80.npz'

def fetch(repo, source, target, repo_type):
    if not target.exists() or target.stat().st_size == 0:
        cached = hf_hub_download(repo, source, repo_type=repo_type)
        shutil.copyfile(cached, target)
    print('Ready:', target.name)

fetch('YongYong/cis6280-stack-world-models-v2',
      'jepa/smwm-c3/model.pt', WM, 'model')
fetch('YongYong/cis6280-stack-world-models-v2',
      'policy/df-smwm-c1-sigma04/policy.pt', POLICY, 'model')
fetch('YongYong/cis6280-stack-datasets-v2',
      'goals/scripted-val100-every80/goals.npz', GOALS, 'dataset')

# Match professor's notebook: policy points at the local world-model checkpoint.
ckpt = torch.load(POLICY, map_location='cpu', weights_only=False)
if ckpt['info']['encoder'] != str(WM.resolve()):
    ckpt['info']['encoder'] = str(WM.resolve())
    torch.save(ckpt, POLICY)
print('Checkpoints and goals ready')


## 4. Define smoothness scoring and planner

In [ ]:
# EXPERIMENT 3: action-smoothness scoring and planner.
import numpy as np
import torch
from assignment.training.data.cache import RAW_ACTION_DIM, STRIDE, BLOCK_ACTION_DIM
from assignment.training.planning.agent import GoalPlanningAgent as OfficialGoalPlanningAgent


def proposed_action_smoothness(blocks, action_mean, action_std, prev_action=None):
    """Mean squared per-joint change in proposed, physical *commands*.

    blocks: [N, H, STRIDE * RAW_ACTION_DIM] normalized policy action blocks.
    action_mean/std: [STRIDE * RAW_ACTION_DIM] arrays from policy checkpoint.
    prev_action: optional [RAW_ACTION_DIM] last executed physical command.
    Returns: [N] lower-is-smoother penalty.
    """
    if blocks.ndim != 3 or blocks.shape[-1] != BLOCK_ACTION_DIM:
        raise ValueError(f'Expected blocks [N,H,{BLOCK_ACTION_DIM}]')
    mean = torch.as_tensor(action_mean, dtype=torch.float32, device=blocks.device)
    std = torch.as_tensor(action_std, dtype=torch.float32, device=blocks.device)
    if mean.numel() != BLOCK_ACTION_DIM or std.numel() != BLOCK_ACTION_DIM:
        raise ValueError('Checkpoint action statistics have wrong shape')
    raw = (blocks.float() * std.reshape(1,1,-1) + mean.reshape(1,1,-1)).clamp(-1,1)
    raw = raw.reshape(blocks.shape[0], -1, RAW_ACTION_DIM)
    delta = raw[:,1:,:] - raw[:,:-1,:]
    if prev_action is not None:
        prev = torch.as_tensor(prev_action, dtype=raw.dtype, device=raw.device)
        if prev.numel() != RAW_ACTION_DIM:
            raise ValueError('prev_action must have RAW_ACTION_DIM elements')
        first_delta = raw[:, :1, :] - prev.reshape(1,1,-1)
        delta = torch.cat([first_delta, delta], dim=1)
    if delta.shape[1] == 0:
        return torch.zeros(blocks.shape[0], dtype=torch.float32, device=blocks.device)
    return delta.square().mean(dim=(1,2))


def combine_terminal_smoothness(terminal_cost, smoothness_cost, beta):
    """Scale smoothness by the mean scores within a candidate set.

    beta=0 produces the original terminal-only scores exactly.
    The scale is a constant for all candidate actions within this decision.
    """
    if beta < 0:
        raise ValueError('beta must be >= 0')
    if terminal_cost.shape != smoothness_cost.shape or terminal_cost.ndim != 1:
        raise ValueError('Both costs must have shape [N]')
    if beta == 0:
        return terminal_cost
    # This avoids mixing potentially unrelated numerical units without scaling.
    ratio = (terminal_cost.detach().mean() /
             smoothness_cost.detach().mean().clamp(min=1e-6))
    return terminal_cost + beta * ratio * smoothness_cost


class SmoothnessPlanningAgent(OfficialGoalPlanningAgent):
    """Original goal planner, changing only the candidate-scoring rule."""

    def __init__(self, policy_path, device='cuda', beta=0.0, **kwargs):
        self.beta = float(beta)
        if self.beta < 0:
            raise ValueError('beta must be nonnegative')
        super().__init__(policy_path, device=device, **kwargs)

    def reset(self, goal_frames=None, init_frame=None, seed=None):
        super().reset(goal_frames=goal_frames, init_frame=init_frame, seed=seed)
        self.prev_raw_action = None
        self.scoring_calls = 0
        self.changed_choices = 0
        self.chosen_smoothness_sum = 0.0
        self.last_total_score = None
        self.last_smoothness = None

    @torch.no_grad()
    def step(self, frame):
        if self.goals is None:
            raise RuntimeError('Call reset(goal_frames, init_frame, seed) before step')

        z = self._encode(frame[None])[0]
        ctx = self.encoder.cfg.context
        self.frames = (self.frames + [z])[-ctx:]
        self.history = (self.history + [(z-self.z_mean)/self.z_std])[-self.policy.cfg.context:]
        goal = self.goals[self.goal_index(z)]

        if not self.queue:
            pctx = [self.history[0]] * (self.policy.cfg.context-len(self.history)) + self.history
            n = self.candidates
            blocks = self.policy.sample(torch.stack(pctx)[None].expand(n,-1,-1),
                                        self.generate, self.n_steps, self.lag, self.generator)
            best = 0
            if n > 1:
                past = self.executed[len(self.executed)-len(self.frames)+1:] if len(self.frames)>1 else []
                acts = (torch.cat([torch.stack(past)[None].expand(n,-1,-1), blocks], dim=1)
                        if past else blocks)
                with self._autocast():
                    pred = self.encoder.rollout(torch.stack(self.frames)[None].expand(n,-1,-1),
                                               self.encoder.action_encoder(acts), self.generate)
                # Same terminal latent cost as the official baseline.
                terminal = (pred[:, -1].float()-goal).square().sum(-1)
                # New term: smoothness of 5x25 raw controls per candidate block.
                smooth = proposed_action_smoothness(blocks, self.a_mean, self.a_std,
                                                    prev_action=self.prev_raw_action)
                total = combine_terminal_smoothness(terminal, smooth, self.beta)
                best = int(total.argmin().item())

                self.last_distance = terminal
                self.last_total_score = total
                self.last_smoothness = smooth
                self.scoring_calls += 1
                self.changed_choices += int(best != int(terminal.argmin().item()))
                self.chosen_smoothness_sum += float(smooth[best].item())

            raw = np.clip(blocks[best, :self.execute].cpu().numpy()*self.a_std+self.a_mean, -1, 1)
            self.queue = list(raw.reshape(-1, STRIDE, RAW_ACTION_DIM))

        block = self.queue.pop(0)
        self.executed.append(torch.from_numpy((block.reshape(-1)-self.a_mean)/self.a_std).to(self.device))
        self.prev_raw_action = np.asarray(block[-1], dtype=np.float32).copy()
        return block

print('Smoothness planner ready. STRIDE =', STRIDE, 'action dimensions =', RAW_ACTION_DIM)


## 5. Mathematical tests (fast)

In [ ]:
# NO simulator or model checkpoints needed for these mathematical tests.
assert STRIDE == 5 and RAW_ACTION_DIM == 25 and BLOCK_ACTION_DIM == 125

N, H = 16, 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mean = torch.zeros(125, device=device)
std = torch.ones(125, device=device)
blocks = torch.randn(N, H, 125, device=device)

# Candidate 0 is stationary; candidate 1 oscillates between +1 and -1.
blocks[0].zero_()
oscillation = (torch.arange(H*STRIDE, device=device) % 2).float()*2-1
blocks[1] = oscillation[:,None].expand(-1, RAW_ACTION_DIM).reshape(H,125)
penalty = proposed_action_smoothness(blocks, mean, std)
assert penalty.shape == (N,)
assert torch.isfinite(penalty).all()
assert penalty[0] == 0
assert penalty[1] > penalty[0]

terminal = torch.linspace(5.0, 10.0, N, device=device)
assert torch.equal(combine_terminal_smoothness(terminal, penalty, beta=0), terminal)
assert torch.isfinite(combine_terminal_smoothness(terminal, penalty, beta=0.2)).all()

# Equal terminal costs -> prefer the smoother candidate (0 vs 1).
pair = torch.tensor([5.0,5.0], device=device)
scores = combine_terminal_smoothness(pair, penalty[:2], beta=0.2)
assert int(scores.argmin()) == 0
print('PASS: physical commands have correct shape')
print('PASS: stationary actions are smoother than oscillating commands')
print('PASS: beta=0 exactly reproduces terminal scoring')
print('PASS: positive beta prefers the smoother candidate when goal costs tie')


## 6. Connect Drive and select the same five scenes

In [ ]:
# Match the SAME five development scenes from Experiment 1.
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/world_model_stacking')
OLD_RESULTS = BASE/'multiscene_eval'
LATE_RESULTS = BASE/'late_horizon_eval'
SMOOTH_RESULTS = BASE/'smoothness_eval'
SMOOTH_RESULTS.mkdir(parents=True, exist_ok=True)

baseline_files = sorted(OLD_RESULTS.glob('scene_*_alpha_000.csv'))
DEV_SCENES = sorted({int(f.stem.split('_')[1]) for f in baseline_files})
assert len(DEV_SCENES) == 5, f'Expected exactly 5 old development scenes, found {DEV_SCENES}'
for seed in DEV_SCENES:
    for tag in ('000','010','025'):
        f = OLD_RESULTS/f'scene_{seed}_alpha_{tag}.csv'
        assert f.exists(), f'Missing Experiment 1 data: {f}'
with np.load(GOALS) as g:
    assert set(DEV_SCENES).issubset(set(map(int,g['seeds'])))

# beta=0 is rerun with instrumented smoothness tracking for a fair baseline.
BETAS = [0.00, 0.05, 0.20, 0.50]
HORIZON = 1795
CANDIDATES = 16
TAU = 0.25
GENERATE = 16
EXECUTE = 1

print('Scenes:', DEV_SCENES)
print('Beta weights:', BETAS)
print('Planned runs:', len(DEV_SCENES)*len(BETAS))
print('Results folder:', SMOOTH_RESULTS)


## 7. Run all scene × beta evaluations (long-running)

In [ ]:
# Complete environment evaluation, matching the previous experiments.
import os, gc, csv, time
from assignment.training.data.cache import STRIDE
from assignment.training.data.signals import row_hand_parts
from assignment.training.data.task import StackTaskEnv, TaskConfig


def evaluate_smoothness(scene_seed, beta):
    scene_seed = int(scene_seed)
    with np.load(GOALS) as gs:
        loc = np.flatnonzero(gs['seeds'] == scene_seed)
        assert len(loc) == 1, f'No unique goals for scene={scene_seed}'
        i = int(loc[0]); n = int(gs['sub_count'][i])
        goal_frames = gs['sub_frames'][i,:n].copy()
        init_frame = gs['init_frames'][i].copy()

    agent = SmoothnessPlanningAgent(
        str(POLICY), 'cuda', candidates=CANDIDATES, generate=GENERATE,
        execute=EXECUTE, tau=TAU, beta=float(beta))
    agent.reset(goal_frames, init_frame, seed=scene_seed)
    env = StackTaskEnv(TaskConfig(target_count=4, horizon=HORIZON,
                                  terminate_on_ground=False), scene_seed)
    env.reset(('init', scene_seed))
    env.env.resolution = 128

    peak=0; final=0; first_placement=None; grasped=False; executed=0
    prev_cmd=None; sum_delta_sq=0.0; num_delta_values=0
    started=time.perf_counter()
    try:
        for step in range(HORIZON):
            if step % STRIDE == 0:
                frame = np.concatenate(list(env.env.render()),axis=1)
                block = agent.step(frame)
            cmd = np.asarray(block[step % STRIDE], dtype=np.float32)
            if prev_cmd is not None:
                sum_delta_sq += float(np.square(cmd-prev_cmd).sum())
                num_delta_values += int(cmd.size)
            prev_cmd = cmd.copy()

            _,_,terminated,truncated,info = env.step(cmd)
            executed = step+1
            final = int(info['placed'])
            peak = max(peak, final)
            if final >= 1 and first_placement is None:
                first_placement = executed
            grasped = grasped or bool((row_hand_parts(env.env.history[-1])[0]>=2).any())
            if terminated or truncated:
                break
    finally:
        env.close()

    duration_min=(time.perf_counter()-started)/60.0
    calls=agent.scoring_calls
    result = {
        'scene_seed': scene_seed,
        'sampler_seed': scene_seed,
        'method': f'smooth_beta_{int(round(beta*100)):03d}',
        'beta': float(beta),
        'candidates': CANDIDATES,
        'tau': TAU,
        'horizon': HORIZON,
        'max_boxes_stacked': peak,
        'final_boxes_stacked': final,
        'stack_loss': peak-final,
        'grasp_success': int(grasped),
        'first_placement_step': first_placement,
        'executed_steps': executed,
        'runtime_minutes': round(duration_min,3),
        'executed_action_smoothness_mse': (sum_delta_sq/num_delta_values
                                            if num_delta_values else 0.0),
        'scoring_calls': calls,
        'candidate_selection_change_rate': (agent.changed_choices/calls if calls else 0.0),
        'mean_selected_predicted_smoothness': (agent.chosen_smoothness_sum/calls if calls else 0.0),
    }
    del agent, env
    return result


jobs = [(s,b) for s in DEV_SCENES for b in BETAS]
for run_idx,(scene,beta) in enumerate(jobs,1):
    tag = int(round(beta*100))
    dst = SMOOTH_RESULTS/f'scene_{scene}_smooth_beta_{tag:03d}.csv'
    if dst.exists():
        print(f'[{run_idx}/{len(jobs)}] SKIP {dst.name}',flush=True)
        continue
    print(f'[{run_idx}/{len(jobs)}] RUN scene={scene}, beta={beta:.2f}',flush=True)
    r=evaluate_smoothness(scene,beta)
    partial = dst.with_suffix('.partial')
    with partial.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(r.keys()));w.writeheader();w.writerow(r)
    os.replace(partial,dst)
    print(f"Saved: peak={r['max_boxes_stacked']}, final={r['final_boxes_stacked']}, "
          f"executed_smoothness={r['executed_action_smoothness_mse']:.5f}",flush=True)

    # Verify beta=0 reproduces the original baseline for that same scene.
    if beta == 0.0:
        old=pd.read_csv(OLD_RESULTS/f'scene_{scene}_alpha_000.csv').iloc[0]
        matching = (int(old['max_boxes_stacked'])==r['max_boxes_stacked'] and
                    int(old['final_boxes_stacked'])==r['final_boxes_stacked'])
        if not matching:
            raise AssertionError(
                f'Baseline parity failed for scene {scene}: '
                f"old peak/final={old['max_boxes_stacked']}/{old['final_boxes_stacked']}; "
                f"new={r['max_boxes_stacked']}/{r['final_boxes_stacked']}. "
                'Stop and diagnose before other beta values.')
        print('PASS: beta=0 reproduces old baseline on scene',scene,flush=True)
    gc.collect()
    torch.cuda.empty_cache()

print('All requested Experiment 3 runs completed.')


## 8. Analyze and plot results

In [ ]:
# Aggregate and compare the SAME five scenes across all experiments.
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rows=[]
for scene in DEV_SCENES:
    for tag,name in [('000','full_baseline_original'),
                     ('010','full_alpha_010'),('025','full_alpha_025')]:
        p=OLD_RESULTS/f'scene_{scene}_alpha_{tag}.csv'
        r=pd.read_csv(p).iloc[0].to_dict();r['method']=name;rows.append(r)
    late=LATE_RESULTS/f'scene_{scene}_late_04_alpha_010.csv'
    if late.exists():
        r=pd.read_csv(late).iloc[0].to_dict();r['method']='late_4';rows.append(r)
    for beta in BETAS:
        tag=int(round(beta*100))
        p=SMOOTH_RESULTS/f'scene_{scene}_smooth_beta_{tag:03d}.csv'
        if p.exists():
            r=pd.read_csv(p).iloc[0].to_dict()
            r['method']=f'smooth_beta_{tag:03d}'
            rows.append(r)

raw=pd.DataFrame(rows)
methods=['full_baseline_original','full_alpha_010','full_alpha_025',
         'late_4']+[f'smooth_beta_{int(round(b*100)):03d}' for b in BETAS]
counts=raw.groupby('scene_seed')['method'].nunique()
complete_seeds=counts[counts==len(methods)].index
if len(complete_seeds)<len(DEV_SCENES):
    print('Some methods are unfinished; analyzing fully paired scenes only:',list(complete_seeds))
paired=raw[raw.scene_seed.isin(complete_seeds)].copy()
if paired.empty:
    raise RuntimeError('No fully paired scenes. Finish Experiment 3 before analysis.')
paired['stack_loss']=paired.max_boxes_stacked-paired.final_boxes_stacked
paired['success_2']=(paired.max_boxes_stacked>=2).astype(float)
paired['success_3']=(paired.max_boxes_stacked>=3).astype(float)
summary=paired.groupby('method').agg(
    scenes=('scene_seed','nunique'),
    mean_peak=('max_boxes_stacked','mean'),
    mean_final=('final_boxes_stacked','mean'),
    mean_stack_loss=('stack_loss','mean'),
    success_ge_2=('success_2','mean'),
    success_ge_3=('success_3','mean'),
    mean_runtime_min=('runtime_minutes','mean'),
    actual_action_smoothness=('executed_action_smoothness_mse','mean'),
    candidate_change_rate=('candidate_selection_change_rate','mean'),
).reindex(methods)
summary[['success_ge_2','success_ge_3']]*=100
print(f'AGGREGATE RESULTS on {len(complete_seeds)} fully paired scenes')
display(summary.round(4))

wide=paired.pivot(index='scene_seed',columns='method',values='max_boxes_stacked')
diffs=pd.DataFrame({
    method+'_minus_baseline':wide[method]-wide['smooth_beta_000']
    for method in methods if method!='smooth_beta_000'
})
print('PAIRED PEAK DIFFERENCES PER SCENE')
display(diffs)

# Direct check the instrumented baseline against old saved baseline.
parity = (wide['full_baseline_original']==wide['smooth_beta_000'])
print('Baseline parity across all paired scenes:',bool(parity.all()))

summary.to_csv(SMOOTH_RESULTS/'experiment3_summary.csv')
paired.to_csv(SMOOTH_RESULTS/'experiment3_paired_runs.csv',index=False)
diffs.to_csv(SMOOTH_RESULTS/'experiment3_paired_differences.csv')

ax=summary[['mean_peak','mean_final']].plot(kind='bar',figsize=(11,4))
ax.set_ylabel('Mean number of connected stacked boxes')
ax.set_xlabel('Scoring method')
ax.set_title(f'Experiments 1–3: {len(complete_seeds)} development scenes')
plt.xticks(rotation=35,ha='right')
plt.tight_layout()
plt.savefig(SMOOTH_RESULTS/'experiment3_comparison.png',dpi=160)
plt.show()

# Isolate actual executed-command smoothness from Experiment 3.
sm=summary.loc[[f'smooth_beta_{int(round(b*100)):03d}' for b in BETAS]]
ax=sm['actual_action_smoothness'].plot(kind='bar',figsize=(7,3))
ax.set_title('Actual executed-action MSE by smoothness weight')
ax.set_ylabel('Mean squared command change')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(SMOOTH_RESULTS/'executed_action_smoothness.png',dpi=160)
plt.show()

print('All results saved to:',SMOOTH_RESULTS)
